In [1]:
import pandas as pd
import numpy as np

# 1. Load your dataset (Replace with your actual file path)
# df = pd.read_csv('credit_card_applications.csv')

# --- DUMMY DATA FOR DEMONSTRATION ---
data = {
    'Applicant_ID': [101, 102, 103, 104, 105, 101],       # ID 101 is a duplicate row
    'Age': [34, 45, 29, 52, 38, 34],
    'Annual_Income': [50000, 80000, 35000, 120000, 65000, 50000],
    'Children_Count': [2, 0, 1, 3, 1, 2],
    'Family_Members': [4, 2, 3, 5, 3, 4],                  # High correlation with Children_Count
    'Work_Phone_Main': [1, 0, 1, 1, 0, 1],
    'Work_Phone_Backup': [1, 0, 1, 1, 0, 1],                # Exact duplicate column
    'Approved': [1, 1, 0, 1, 0, 1]
}
df = pd.DataFrame(data)
print(f"Original shape: {df.shape}")


# 2. Drop Duplicate Rows
# Removes perfectly identical rows to prevent training bias
df = df.drop_duplicates()
print(f"Shape after dropping duplicate rows: {df.shape}")


# 3. Drop Exact Duplicate Columns (Features)
# Transposes the dataframe to scan and delete completely identical feature blocks
duplicate_features = df.T.duplicated()
exact_dup_cols = duplicate_features[duplicate_features == True].index.tolist()

df = df.drop(columns=exact_dup_cols)
print(f"Dropped exact duplicate features: {exact_dup_cols}")


# 4. Drop Highly Correlated (Redundant) Features
# Calculates Pearson correlation to drop features carrying the same information
# (e.g., dropping 'Children_Count' because 'Family_Members' already captures it)
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr().abs()

# Select upper triangle of correlation matrix
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Find features with a correlation higher than a 0.85 threshold
high_corr_cols = [column for column in upper_tri.columns if any(upper_tri[column] > 0.85)]

df = df.drop(columns=high_corr_cols)
print(f"Dropped highly correlated redundant features: {high_corr_cols}")
print(f"Final cleaned dataset shape: {df.shape}")


Original shape: (6, 8)
Shape after dropping duplicate rows: (5, 8)
Dropped exact duplicate features: ['Work_Phone_Backup']
Dropped highly correlated redundant features: ['Annual_Income', 'Family_Members']
Final cleaned dataset shape: (5, 5)
